# ML Coding — Vectorized Softmax — AI Lab Instructor Notebook

*NLP & Transformers · Hard*



6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Vectorized Softmax — Student Lab

Implement numerically stable softmax + cross-entropy and fully vectorized gradients.
Then verify with finite-difference gradient checking.

## Section 0 — Toy dataset
We generate a simple 3-class dataset in 2D.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_toy(n_per_class: int = 80, k: int = 3, seed: int = 0):
    rng = np.random.default_rng(seed)
    Xs = []
    ys = []
    centers = np.array([[2.0, 0.0], [-2.0, 0.0], [0.0, 2.0]])
    for c in range(k):
        Xc = centers[c] + 0.8 * rng.standard_normal((n_per_class, 2))
        yc = np.full((n_per_class,), c, dtype=np.int64)
        Xs.append(Xc)
        ys.append(yc)
    X = np.vstack(Xs)
    y = np.concatenate(ys)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]

X, y = make_toy()
n, d = X.shape
k = int(y.max()) + 1
print('X', X.shape, 'y', y.shape, 'k', k)
check('labels_range', int(y.min()) == 0 and int(y.max()) == k-1)

**Why this works.** # Vectorized Softmax — Instructor Lab

Complete solutions + short rationales. Fully vectorized, stable, and gradient-checked.

## Toy dataset

## Stable log-softmax
Rationale: `log_softmax` via log-sum-exp avoids overflow and is the standard primitive.

## Task 2: Section 1 — Stable softmax
*Section: Section 1*

## Section 1 — Stable softmax

### Task 1.1
Implement `log_softmax(Z)` for `Z` shape (n,k).
Hint: subtract row-wise max; use log-sum-exp.

**Hints**
- subtract row-wise max; use log-sum-exp.

In [ ]:
def log_softmax(Z: np.ndarray) -> np.ndarray:
    Zm = Z - np.max(Z, axis=1, keepdims=True)
    lse = np.log(np.sum(np.exp(Zm), axis=1, keepdims=True))
    return Zm - lse

def softmax(Z: np.ndarray) -> np.ndarray:
    return np.exp(log_softmax(Z))

Z = rng.standard_normal((4, 5)) * 50
p = softmax(Z)
check('rowsum1', float(np.max(np.abs(p.sum(axis=1) - 1.0))) < 1e-8)
check('finite', np.isfinite(p).all())

In [ ]:
# Checks
check('rowsum1', float(np.max(np.abs(p.sum(axis=1) - 1.0))) < 1e-8)
check('finite', np.isfinite(p).all())

## Task 3: Section 2 — Cross-entropy loss + logits gradient
*Section: Section 2*

## Section 2 — Cross-entropy loss + gradient wrt logits

### Task 2.1
Implement `xent_loss_and_grad_logits(logits, y)` returning:
- `loss`: average cross-entropy
- `dlogits`: gradient with same shape as logits

Hint: if `P = softmax(logits)`, then `dlogits = (P - one_hot(y))/n`.

**Hints**
- if `P = softmax(logits)`, then `dlogits = (P - one_hot(y))/n`.

In [ ]:
def xent_loss_and_grad_logits(logits: np.ndarray, y: np.ndarray):
    n = logits.shape[0]
    ls = log_softmax(logits)
    loss = -np.mean(ls[np.arange(n), y])
    P = np.exp(ls)
    dlogits = P
    dlogits[np.arange(n), y] -= 1.0
    dlogits /= n
    return float(loss), dlogits

logits = rng.standard_normal((10, 3))
y_small = rng.integers(0, 3, size=(10,), dtype=np.int64)
loss, dlogits = xent_loss_and_grad_logits(logits, y_small)
check('shape', dlogits.shape == logits.shape)
check('row_sum0', float(np.max(np.abs(dlogits.sum(axis=1)))) < 1e-8)

In [ ]:
# Checks
check('shape', dlogits.shape == logits.shape)
check('row_sum0', float(np.max(np.abs(dlogits.sum(axis=1)))) < 1e-8)

**Why this works.** ## Cross-entropy + gradient wrt logits
Rationale: for integer labels, loss uses `log_softmax` indexing; gradient is `P - one_hot`.

## Task 4: Section 3 — Softmax regression gradients
*Section: Section 3*

## Section 3 — Softmax regression (W,b) gradients

logits = X @ W + b

### Task 3.1
Implement `loss_and_grads(W,b,X,y)` returning loss and gradients.
Constraints: fully vectorized.

In [ ]:
def loss_and_grads(W: np.ndarray, b: np.ndarray, X: np.ndarray, y: np.ndarray):
    logits = X @ W + b
    loss, dlogits = xent_loss_and_grad_logits(logits, y)
    dW = X.T @ dlogits
    db = dlogits.sum(axis=0)
    return loss, {'W': dW, 'b': db}

W = 0.01 * rng.standard_normal((d, k))
b = np.zeros((k,), dtype=np.float64)
loss, grads = loss_and_grads(W, b, X[:20], y[:20])
print('loss', loss)
check('shapes', grads['W'].shape == W.shape and grads['b'].shape == b.shape)

In [ ]:
# Checks
check('shapes', grads['W'].shape == W.shape and grads['b'].shape == b.shape)

**Why this works.** ## Softmax regression gradients
Rationale: `dW = X^T dlogits`, `db = sum(dlogits)` via vectorization.

## Task 5: Section 4 — Gradient checking
*Section: Section 4*

## Section 4 — Gradient checking (finite differences)

### Task 4.1
Implement a finite-difference check for a few random coordinates of `W` and `b`.
Report the max relative error.

In [ ]:
$f

In [ ]:
# Checks
check('gradcheck_ok', err < 1e-5)

**Why this works.** ## Gradient check
Rationale: catches sign/axis/broadcasting errors. Use small h and compare relative error.

## Task 6: Section 5 — Train loop
*Section: Section 5*

## Section 5 — Train loop

### Task 5.1
Train softmax regression and show loss decreases.

In [ ]:
def sgd_train(X: np.ndarray, y: np.ndarray, k: int, steps: int = 250, lr: float = 0.8, seed: int = 0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    W = 0.01 * rng.standard_normal((d, k))
    b = np.zeros((k,), dtype=np.float64)
    losses = []
    for t in range(steps):
        loss, grads = loss_and_grads(W, b, X, y)
        W -= lr * grads['W']
        b -= lr * grads['b']
        losses.append(loss)
        if (t+1) % 50 == 0:
            print('step', t+1, 'loss', round(loss, 4))
    return W, b, losses

Wf, bf, losses = sgd_train(X, y, k=k)
check('train_loss_down', losses[-1] < losses[0])

In [ ]:
# Checks
check('train_loss_down', losses[-1] < losses[0])

**Why this works.** ## Train loop
Rationale: an implementation can pass shape checks yet still be wrong; loss should decrease.